# NB 01: Hugging Face Ecosystem Tour

**Goal:** Understand how the HF ecosystem connects to our vol surface data.

This notebook covers the four core HF libraries we'll use throughout the project:

| Library | What it does | Our use case |
|---------|-------------|--------------|
| `datasets` | Standardised data loading, transforms, splits | Wrap SQLite vol data for HF Trainer API |
| `huggingface_hub` | Search/download models from the Hub | Find finance-specific pre-trained models |
| `transformers` | Model inference and training | FinBERT sentiment, Transformer vol prediction |
| `diffusers` | Diffusion model training/generation | DDPM generative vol surfaces (NB 05) |

**Data source:** 429,936 vol surface grid points from `rl_hedging_comparison` (SPX, 2010-2026, Marquee).

## Setup

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from datasets import Dataset

from hf_volsurf.data.loaders import VolSurfaceDataLoader
from hf_volsurf.data.hf_dataset import create_pointwise_dataset, create_image_dataset
from hf_volsurf.utils.vol_math import TENOR_ORDER, STRIKE_GRID, tenor_to_years

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print("=== NB 01: HF Ecosystem Tour ===\n")

c:\source\repos\ale\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


PROJECT_ROOT: c:\source\repos\ale\huggin-face\hugging-face-learning
=== NB 01: HF Ecosystem Tour ===



## 1. Data inventory

Our SQLite database (`hf_volsurf.db`, 69 MB) was copied from the RL hedging project. Five tables covering 16 years of SPX market data — the vol surface alone has ~430k grid points.

In [3]:
loader = VolSurfaceDataLoader(PROJECT_ROOT / "data" / "db" / "hf_volsurf.db")
summary = loader.get_data_summary()
print("Data summary:")
for table, info in summary.items():
    print(f"  {table:25s} {info['rows']:>10,} rows  [{info['min_date']} to {info['max_date']}]")

Data summary:
  spx_vol_surface              429,936 rows  [2010-01-04 to 2026-01-05]
  spx_spot_prices                4,034 rows  [2010-01-04 to 2026-01-05]
  spx_dividend_yield             5,849 rows  [2010-01-01 to 2026-01-05]
  ois_curve                     40,943 rows  [2010-01-01 to 2026-01-05]
  vix_data                       2,798 rows  [2015-01-02 to 2026-01-06]


## 2. Pointwise HF Dataset

`create_pointwise_dataset()` wraps our SQLite data into an HF `Dataset` object. Each row is one observation: *(date, moneyness, tenor, implied_vol, spot, vix)*.

For 2020 alone this gives **26,832 rows** (258 trading days × 104 grid points). The full dataset has 429k rows.

This "pointwise" format is ideal for **regression models** — predict `implied_vol` from `(moneyness, tenor, spot, vix)` features.

In [4]:
print("\n--- Creating pointwise HF Dataset ---")
ds = create_pointwise_dataset(
    db_path=PROJECT_ROOT / "data" / "db" / "hf_volsurf.db",
    start="2020-01-01",
    end="2020-12-31",
)
print(f"Dataset: {ds}")
print(f"Features: {ds.features}")
print(f"First row: {ds[0]}")


--- Creating pointwise HF Dataset ---
Dataset: Dataset({
    features: ['date', 'moneyness', 'tenor_years', 'implied_vol', 'spot', 'vix'],
    num_rows: 26832
})
Features: {'date': Value('string'), 'moneyness': Value('float64'), 'tenor_years': Value('float64'), 'implied_vol': Value('float64'), 'spot': Value('float64'), 'vix': Value('float64')}
First row: {'date': '2020-01-02', 'moneyness': 0.7, 'tenor_years': 0.08333333333333333, 'implied_vol': 0.37005760535826093, 'spot': 3259.0, 'vix': 12.47}


## 3. HF Datasets API — filter, map, split

Key operations that "just work" on our vol data:

- **`filter`** — select ATM options (`moneyness == 1.0`): 2,064 rows from 26,832 (8 tenors × 258 days)
- **`map`** — compute derived features like total variance $w = \sigma^2 \cdot T$, which must be non-decreasing in $T$ for calendar-spread arbitrage freedom
- **`train_test_split`** — random 80/20 split (for proper time-series work we use chronological splits in NB 02)

In [5]:
print("\n--- HF Dataset operations ---")

# Filter: only ATM options (moneyness = 1.0)
atm_ds = ds.filter(lambda x: x["moneyness"] == 1.0)
print(f"ATM subset: {len(atm_ds)} rows (from {len(ds)} total)")

# Map: add a column (total variance = iv^2 * tenor)
def add_total_variance(example):
    example["total_variance"] = example["implied_vol"] ** 2 * example["tenor_years"]
    return example

ds_with_tv = atm_ds.map(add_total_variance)
print(f"Added total_variance column: {ds_with_tv.column_names}")
print(f"Sample total_variance: {ds_with_tv[0]['total_variance']:.6f}")

# Train/test split
split = ds.train_test_split(test_size=0.2, seed=42)
print(f"Train: {len(split['train'])}, Test: {len(split['test'])}")


--- HF Dataset operations ---


Filter:   0%|          | 0/26832 [00:00<?, ? examples/s]

ATM subset: 2064 rows (from 26832 total)


Map:   0%|          | 0/2064 [00:00<?, ? examples/s]

Added total_variance column: ['date', 'moneyness', 'tenor_years', 'implied_vol', 'spot', 'vix', 'total_variance']
Sample total_variance: 0.000880
Train: 21465, Test: 5367


## 4. Image dataset — surfaces as 2D grids

`create_image_dataset()` reshapes each day's vol surface into an **(8, 13)** array — 8 tenors × 13 strikes. This is the format the diffusion model (NB 05) will consume: each surface is a tiny "image" where pixel intensity = implied volatility.

**4,134 complete daily surfaces** available. First surface (2010-01-04): IV ranges from 17.6% to 49.7%.

In [6]:
print("\n--- Creating image HF Dataset ---")
img_ds = create_image_dataset(db_path=PROJECT_ROOT / "data" / "db" / "hf_volsurf.db")
print(f"Image dataset: {len(img_ds)} surfaces")
print(f"Features: {img_ds.column_names}")
surface_0 = np.array(img_ds[0]["surface"])
print(f"First surface shape: {surface_0.shape}, min IV: {surface_0.min():.4f}, max IV: {surface_0.max():.4f}")


--- Creating image HF Dataset ---
Image dataset: 4134 surfaces
Features: ['date', 'surface', 'spot', 'vix', 'regime']
First surface shape: (8, 13), min IV: 0.1761, max IV: 0.4968


## 5. HuggingFace Hub — model discovery

`HfApi.list_models()` searches the Hub by keyword. **ProsusAI/finbert** dominates the finance NLP space with ~4.8M downloads. It's a BERT model fine-tuned on financial corpus data (analyst reports, earnings calls, financial news) for 3-class sentiment: positive / negative / neutral.

In [7]:
print("\n--- HuggingFace Hub: searching for finance models ---")
from huggingface_hub import HfApi

api = HfApi()
models = list(api.list_models(search="finbert", sort="downloads", limit=5))
print(f"Found {len(models)} FinBERT models:")
for m in models:
    print(f"  {m.id:50s} downloads={m.downloads:>10,}")


--- HuggingFace Hub: searching for finance models ---
Found 5 FinBERT models:
  ProsusAI/finbert                                   downloads= 4,848,805
  yiyanghkust/finbert-tone                           downloads=   925,536
  lucas-leme/FinBERT-PT-BR                           downloads=   148,927
  yiyanghkust/finbert-tone-chinese                   downloads=   135,030
  snunlp/KR-FinBert-SC                               downloads=    80,611


## 6. FinBERT inference — 3 lines of code

The HF `pipeline()` API is the fastest path from zero to inference. Three lines: create pipeline, pass text, get labels.

**Observations:**
- "Fed raises rates by 75bps" → **neutral** (0.76). FinBERT treats rate hikes as factual, not directional — interesting.
- "Tech stocks rally..." → **negative** (0.88). Surprising — likely because the word "inflation" dominates the token attention.
- "S&P 500 hits all-time high" → **positive** (0.87). Correctly identified.

FinBERT's domain-specific training matters: a general-purpose sentiment model would likely miss the nuance that "rate hike" is neutral for the financial audience while "market concerns" is negative.

In [8]:
print("\n--- FinBERT sentiment inference ---")
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="ProsusAI/finbert")

headlines = [
    "Federal Reserve raises interest rates by 75 basis points",
    "Tech stocks rally as inflation data comes in below expectations",
    "Goldman Sachs reports record quarterly earnings",
    "Market volatility spikes amid banking sector concerns",
    "S&P 500 hits all-time high on strong economic data",
]

results = sentiment(headlines)
print(f"\n{'Headline':65s} {'Label':10s} {'Score'}")
print("-" * 90)
for headline, result in zip(headlines, results):
    print(f"{headline:65s} {result['label']:10s} {result['score']:.4f}")


--- FinBERT sentiment inference ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Headline                                                          Label      Score
------------------------------------------------------------------------------------------
Federal Reserve raises interest rates by 75 basis points          neutral    0.7643
Tech stocks rally as inflation data comes in below expectations   negative   0.8759
Goldman Sachs reports record quarterly earnings                   positive   0.6588
Market volatility spikes amid banking sector concerns             negative   0.8665
S&P 500 hits all-time high on strong economic data                positive   0.8706


## Summary

| What we tested | Result |
|---------------|--------|
| `datasets.Dataset` on vol data | 26,832 rows (2020), filter/map/split all work |
| Image dataset for diffusion | 4,134 daily surfaces as (8, 13) grids |
| HuggingFace Hub search | 5 FinBERT variants found, ProsusAI/finbert has 4.8M downloads |
| `pipeline()` inference | 3 lines of code: load model → pass text → get labels |

**Next:** NB 02 explores the vol surface data in depth — 3D plots, skew dynamics, and train/val/test split.

In [9]:
print("\n=== NB 01 Complete ===")
print("Key takeaways:")
print("  - datasets.Dataset wraps our vol surface data for HF APIs")
print("  - filter(), map(), train_test_split() work out of the box")
print("  - HuggingFace Hub has finance-specific models (FinBERT)")
print("  - pipeline() makes inference trivial (3 lines of code)")


=== NB 01 Complete ===
Key takeaways:
  - datasets.Dataset wraps our vol surface data for HF APIs
  - filter(), map(), train_test_split() work out of the box
  - HuggingFace Hub has finance-specific models (FinBERT)
  - pipeline() makes inference trivial (3 lines of code)
